# TemporalRec: Timestamp-Basket Tradeoff Audit

## Tradeoff found after filtering

Excluding every same-visitor, same-timestamp group preserves a strict one-item-at-a-time order, but removed 3,728 purchases: 16.6% of transaction events. A basket-aware approach keeps these purchases, but treats items at the same timestamp as unordered multiple targets.

This notebook audits that tradeoff. It does not replace the current 5-core dataset, create model sequences, or invent an order inside a basket.


## Time blocks

A time block means all events from one visitor at one exact timestamp.

A block with one row is an ordinary interaction. A tied block has two or more rows at that timestamp. A transaction basket has multiple purchased items sharing one visitor, timestamp, and transaction ID.

For a future basket-aware example, context may use only earlier time blocks. Every distinct item in the next block is a valid target. The order within a block is unknown.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "dataset" / "events.csv").exists()),
    None,
)
assert PROJECT_ROOT is not None, "Open this notebook inside the Recommendation project."

INPUT_RUN_TAG = "notebook_run_2026_10_13"
RAW_FILE = PROJECT_ROOT / "dataset" / "events.csv"
EXACT_DEDUPED_FILE = (
    PROJECT_ROOT / "data" / "interim" / INPUT_RUN_TAG / "events_exact_deduplicated.csv"
)
for path in [RAW_FILE, EXACT_DEDUPED_FILE]:
    assert path.exists(), f"Missing input: {path}"

RUN_TAG = "timestamp_basket_audit_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
REPORT_DIR = PROJECT_ROOT / "reports" / RUN_TAG
REPORT_DIR.mkdir(parents=True, exist_ok=False)

FIELDS = ["timestamp", "visitorid", "event", "itemid", "transactionid"]
EVENT_TYPES = ["view", "addtocart", "transaction"]
DTYPES = {
    "timestamp": "int64",
    "visitorid": "int64",
    "event": "string",
    "itemid": "int64",
    "transactionid": "Int64",
}
MODEL_TIME_FIELDS = ["visitorid", "timestamp"]
assert "source_row_number" not in MODEL_TIME_FIELDS

print("Reports:", REPORT_DIR)


## 1. Reconstruct exact-deduplicated data

This verifies that Notebook 01's saved cleaned file matches the raw data after exact duplicate removal. Source row number is audit metadata only. It is never used to order a visitor's events.


In [ ]:
raw = pd.read_csv(RAW_FILE, dtype=DTYPES)
assert raw.columns.tolist() == FIELDS
assert raw[["timestamp", "visitorid", "event", "itemid"]].notna().all().all()
assert set(raw["event"]) <= set(EVENT_TYPES)

exact_deduplicated = raw.drop_duplicates(FIELDS, keep="first").copy()
saved_exact_deduplicated = pd.read_csv(EXACT_DEDUPED_FILE, dtype=DTYPES)
pd.testing.assert_frame_equal(
    exact_deduplicated.reset_index(drop=True),
    saved_exact_deduplicated.reset_index(drop=True),
)
del saved_exact_deduplicated

exact_deduplicated["source_row_number"] = exact_deduplicated.index + 2
print(f"Raw rows: {len(raw):,}")
print(f"Exact-deduplicated rows: {len(exact_deduplicated):,}")
print("Notebook 01 output matches.")


## 2. Classify tied blocks

same_item_tie: all rows refer to one item.

transaction_basket: all rows are transactions, every transaction ID is present, one transaction ID is shared, and the block has multiple items.

other_multi_item_tie: multiple items but not a clean transaction basket.

unexpected_ambiguous_tie: a tied transaction group with missing or conflicting transaction IDs.


In [ ]:
block_sizes = exact_deduplicated.groupby(MODEL_TIME_FIELDS)["itemid"].transform("size")
tied_rows = exact_deduplicated.loc[block_sizes.gt(1)].copy()
tied_rows["time_block_id"] = (
    tied_rows.groupby(MODEL_TIME_FIELDS, sort=True).ngroup() + 1
)

def join_unique(values):
    return " + ".join(sorted(str(value) for value in pd.unique(values)))

tie_group_summary = (
    tied_rows.groupby("time_block_id", as_index=False)
    .agg(
        visitorid=("visitorid", "first"),
        timestamp=("timestamp", "first"),
        event_rows=("itemid", "size"),
        distinct_items=("itemid", "nunique"),
        event_types=("event", join_unique),
        transaction_rows=("event", lambda values: int(values.eq("transaction").sum())),
        nonblank_transaction_ids=("transactionid", lambda values: int(values.notna().sum())),
        distinct_transaction_ids=("transactionid", lambda values: int(values.dropna().nunique())),
    )
)

def classify_group(row):
    if row["distinct_items"] == 1:
        return "same_item_tie"
    if (
        row["transaction_rows"] == row["event_rows"]
        and row["nonblank_transaction_ids"] == row["event_rows"]
        and row["distinct_transaction_ids"] == 1
    ):
        return "transaction_basket"
    if row["transaction_rows"] == row["event_rows"]:
        return "unexpected_ambiguous_tie"
    return "other_multi_item_tie"

tie_group_summary["classification"] = tie_group_summary.apply(classify_group, axis=1)
tie_group_summary = tie_group_summary[
    [
        "time_block_id", "classification", "visitorid", "timestamp",
        "event_rows", "distinct_items", "event_types", "transaction_rows",
        "nonblank_transaction_ids", "distinct_transaction_ids",
    ]
].sort_values(["classification", "visitorid", "timestamp"], kind="stable")

assert len(tied_rows) == 4769
assert tie_group_summary["event_rows"].sum() == len(tied_rows)
assert tie_group_summary.groupby("time_block_id")[MODEL_TIME_FIELDS].nunique().eq(1).all().all()
assert tie_group_summary["classification"].eq("same_item_tie").sum() == 11

display(
    tie_group_summary["classification"]
    .value_counts()
    .rename_axis("classification")
    .reset_index(name="time_blocks")
)
display(tie_group_summary.head(10))


## 3. Save transaction-basket and Excel audits

Source-row number appears in these files only to trace a row back to the raw file. It does not mean one basket item happened before another.


In [ ]:
basket_ids = tie_group_summary.loc[
    tie_group_summary["classification"].eq("transaction_basket"),
    "time_block_id",
]
transaction_basket_audit = (
    tied_rows.loc[tied_rows["time_block_id"].isin(basket_ids)]
    .merge(
        tie_group_summary[
            ["time_block_id", "classification", "event_rows", "distinct_items"]
        ],
        on="time_block_id",
        how="left",
        validate="many_to_one",
    )
    .sort_values(["time_block_id", "source_row_number"], kind="stable")
)
transaction_basket_audit = transaction_basket_audit[
    [
        "time_block_id", "classification", "event_rows", "distinct_items",
        "source_row_number", "visitorid", "timestamp", "transactionid",
        "event", "itemid",
    ]
]

example_ids = tie_group_summary["time_block_id"].head(200)
tie_group_examples = (
    tied_rows.loc[tied_rows["time_block_id"].isin(example_ids)]
    .merge(
        tie_group_summary[
            ["time_block_id", "classification", "event_rows", "distinct_items"]
        ],
        on="time_block_id",
        how="left",
        validate="many_to_one",
    )
    .sort_values(["time_block_id", "source_row_number"], kind="stable")
)
tie_group_examples = tie_group_examples[transaction_basket_audit.columns]

assert transaction_basket_audit["event"].eq("transaction").all()
assert transaction_basket_audit["transactionid"].notna().all()
assert transaction_basket_audit["time_block_id"].nunique() == len(basket_ids)
assert len(transaction_basket_audit) == 3728

tie_group_summary.to_csv(REPORT_DIR / "tie_group_summary.csv", index=False)
transaction_basket_audit.to_csv(REPORT_DIR / "transaction_basket_audit.csv", index=False)
tie_group_examples.to_csv(REPORT_DIR / "tie_group_examples.csv", index=False)

print(f"Transaction baskets: {len(basket_ids):,}")
print(f"Transaction rows in baskets: {len(transaction_basket_audit):,}")
display(transaction_basket_audit.head(10))


## 4. Compare current exclusion with basket-aware retention

The current policy removes every tied row. The basket-aware policy retains every exact-deduplicated row but preserves ties as unordered time blocks. This comparison happens before the 5-core filter.


In [ ]:
tied_mask = block_sizes.gt(1)
current_exclusion = exact_deduplicated.loc[~tied_mask].copy()
basket_aware = exact_deduplicated.copy()

def policy_statistics(policy, frame):
    rows = []
    for event_type in ["all_events", *EVENT_TYPES]:
        scoped = frame if event_type == "all_events" else frame.loc[frame["event"].eq(event_type)]
        rows.append(
            {
                "policy": policy,
                "event_type": event_type,
                "event_rows": len(scoped),
                "visitors": scoped["visitorid"].nunique(),
                "items": scoped["itemid"].nunique(),
                "time_blocks": scoped[MODEL_TIME_FIELDS].drop_duplicates().shape[0],
            }
        )
    return rows

tie_policy_comparison = pd.DataFrame(
    policy_statistics("current_exclusion", current_exclusion)
    + policy_statistics("basket_aware", basket_aware)
)
basket_counts = (
    tie_policy_comparison.loc[tie_policy_comparison["policy"].eq("basket_aware")]
    .set_index("event_type")["event_rows"]
)
tie_policy_comparison["rows_lost_vs_basket_aware"] = (
    tie_policy_comparison["event_type"].map(basket_counts)
    - tie_policy_comparison["event_rows"]
)

assert len(basket_aware) - len(current_exclusion) == 4769
current_transactions = int(current_exclusion["event"].eq("transaction").sum())
basket_transactions = int(basket_aware["event"].eq("transaction").sum())
assert basket_transactions - current_transactions == 3728

tie_policy_comparison.to_csv(REPORT_DIR / "tie_policy_comparison.csv", index=False)
display(tie_policy_comparison)


## 5. Record the future basket-aware contract

This is a rule sheet for later sequence and evaluation work. It is not a training dataset.


In [ ]:
basket_contract = pd.DataFrame(
    [
        ["Time block", "One visitor plus one exact timestamp."],
        ["History", "Use only blocks with timestamps strictly earlier than the target block."],
        ["Target", "Every distinct item in the target block is a valid positive target."],
        ["Within-block order", "Unknown. Never use source-row order, item ID, or transaction ID to create item order."],
        ["Evaluation implication", "Later ranking evaluation must support multiple positive items in one target block."],
        ["Model implication", "Later design must represent earlier baskets without fabricating internal item order."],
    ],
    columns=["rule", "definition"],
)
basket_contract.to_csv(REPORT_DIR / "basket_contract.csv", index=False)
display(basket_contract)


## Conclusion and Excel checks

The exclusion policy is simple for strict one-item next-item sequences, but it loses many recorded purchases. The basket-aware audit retains those purchases truthfully. It creates a multi-positive target problem because the next time block can contain several valid items.

Do not create basket-aware sequences until the later model and evaluation design is documented.

For Excel:

1. Open tie_group_summary.csv. Filter classification to same_item_tie. There should be 11 groups.
2. Filter classification to transaction_basket. Check its time_block_id values in transaction_basket_audit.csv.
3. In each transaction basket, visitor ID, timestamp, and transaction ID match. Item IDs differ.
4. In tie_policy_comparison.csv, basket-aware should retain 3,728 more transaction rows.
5. basket_contract.csv contains the rules for the next design step.
